# Three-model comparison on a Kaggle GPU

Runs the same attacks on **qwen3:8b, llama3.1:8b and mistral:7b**.

Before running: **Settings → Accelerator → GPU T4 x1** and **Settings → Internet → On** (needs a phone-verified Kaggle account). Then *Run All*. At the end, download `results-models.zip` from the Output panel.

Stage 1 (screening): every attack with no defense, plus the normal tasks, on each model. Stage 2: three key defenses on each model. GPU results go to their own file and are never mixed with the earlier CPU runs.

In [ ]:
MODELS = ['qwen3:8b', 'llama3.1:8b', 'mistral:7b']
DEFENSES = 'spotlight,policy,spotlight+policy+guard'
AUDIT = 'results/audit-gpu-models.jsonl'
BRANCH = 'study/full-sweep'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q --branch $BRANCH https://github.com/tulasinayak/understanding-prompt-injection-mcp.git /kaggle/working/testbed
%cd /kaggle/working/testbed
!git log --oneline -1

In [ ]:
!pip install -q uv
!uv sync -q && echo 'python environment ready'

In [ ]:
# Ollama: install, start the server in the background, pull the three models
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, urllib.request
server = subprocess.Popen(['ollama', 'serve'], stdout=open('/kaggle/working/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
for m in MODELS:
    !ollama pull {m} 2>&1 | tail -1
!ollama list

In [ ]:
# the AgentDojo attack set is generated locally (it is not stored in the repository)
!uv run python tools/gen_agentdojo.py
!uv run pytest -q 2>&1 | tail -1

In [ ]:
# one quick run per model, to check each one can use the tools
for m in MODELS:
    print('=====', m)
    !uv run python -m harness.run --scenario exfil-ignore-vpn --defense none --model {m} --audit results/scratch-gpu.jsonl | tail -4

In [ ]:
# Stage 1: screening - every attack with no defense, plus the normal tasks
for m in MODELS:
    !uv run python -m harness.sweep --defenses none --model {m} --audit {AUDIT}

In [ ]:
# Stage 2: the key defenses on each model
for m in MODELS:
    !uv run python -m harness.sweep --defenses {DEFENSES} --model {m} --audit {AUDIT}

In [ ]:
!uv run python -m harness.report {AUDIT}
!sed -n '/## Models compared/,/## Attacks grouped/p' results/REPORT-gpu-models.md
!cd results && zip -q /kaggle/working/results-models.zip audit-gpu-models.jsonl REPORT-gpu-models.md
print('Download results-models.zip from the Output panel.')